# Lab

Plan a facilities day: room schedules, cable links, and delivery routes. Each cell is standalone; carry methods forward explicitly. State each objective and units. Use Run and sign in before saving work.

## 1. Choose Compatible Room Bookings

```{index} 1. Choose Compatible Room Bookings
```

Implement earliest-finish scheduling under the positive-duration half-open interval contract. Use the supplied bookings and print selected IDs/count. Explain the exchange step, not only feasibility.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",1,4),new MeetingSchedule.Meeting("B",3,5),new MeetingSchedule.Meeting("C",0,6),new MeetingSchedule.Meeting("D",5,7),new MeetingSchedule.Meeting("E",8,9)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
var meetings=new[]{new MeetingSchedule.Meeting("A",1,4),new MeetingSchedule.Meeting("B",3,5),new MeetingSchedule.Meeting("C",0,6),new MeetingSchedule.Meeting("D",5,7),new MeetingSchedule.Meeting("E",8,9)};
var chosen=MeetingSchedule.Select(meetings);
Console.WriteLine($"IDs={string.Join(",",chosen.Select(m=>m.Id))}, count={chosen.Length}");
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


IDs=A,D,E, count=3


## 2. Challenge the Objective

```{index} 2. Challenge the Objective
```

Give the three bookings below values and compare the count-based greedy selection with booking C alone. Print both values and explain why the count proof cannot establish maximum value.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",0,2,4),new MeetingSchedule.Meeting("B",2,4,4),new MeetingSchedule.Meeting("C",0,4,9)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
var meetings=new[]{new MeetingSchedule.Meeting("A",0,2,4),new MeetingSchedule.Meeting("B",2,4,4),new MeetingSchedule.Meeting("C",0,4,9)};
var chosen=MeetingSchedule.Select(meetings);
Console.WriteLine($"greedy count={chosen.Length}, value={chosen.Sum(m=>m.Value)}; C count=1, value=9");
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


greedy count=2, value=8; C count=1, value=9


## 3. Build the Cable Forest

```{index} 3. Build the Cable Forest
```

Implement Kruskal with cycle rejection and union-find. Trace each edge decision, preserve Remote as an isolate, and report components, edge count, and long total. Explain the cut argument and why this is a forest.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var edges=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var edges=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var forest=GreedyGraph.Kruskal(vertices,edges,Console.WriteLine);
Console.WriteLine($"cost={forest.Cost}, edges={forest.Edges.Length}, components={forest.Components}");
if(forest.Edges.Length!=vertices.Length-forest.Components)throw new Exception("Forest count failed.");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


accept HQ-Shop (2): components=4
accept Shop-Lab (3): components=3
accept Lab-Depot (4): components=2
reject Shop-Depot (5): cycle
reject HQ-Lab (6): cycle
cost=9, edges=3, components=2


## 4. Find Delivery Routes

```{index} 4. Find Delivery Routes
```

Implement directed nonnegative Dijkstra with stale-entry skipping and predecessor recording. Print HQ, Depot, and Remote routes/distances. An unreachable destination must have an empty path, not a fabricated singleton.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"HQ",Console.WriteLine);
foreach(string target in new[]{"HQ","Depot","Remote"})Console.WriteLine($"{target}: {(routes.Distance[target]?.ToString()??"unreachable")}, [{string.Join("->",routes.PathTo(target))}]");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


settle HQ (0)
relax HQ->Shop: 2
relax HQ->Lab: 6
settle Shop (2)
relax Shop->Lab: 5
relax Shop->Depot: 7
settle Lab (5)
skip stale Lab (6)
settle Depot (7)
HQ: 0, [HQ]
Depot: 7, [HQ->Shop->Depot]
Remote: unreachable, []


## 5. Explain a Planning Change

```{index} 5. Explain a Planning Change
```

Add a directed HQ-to-Remote delivery arc with weight zero, leaving the cable edge set unchanged. Report new delivery distance/path and cable components. Verify that route reachability does not make the cable network connected.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var edges=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var edges=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var forest=GreedyGraph.Kruskal(vertices,edges);
var routes=GreedyGraph.Dijkstra(vertices,arcs.Append(new GreedyGraph.Edge("HQ","Remote",0)).ToArray(),"HQ");
Console.WriteLine($"Remote delivery={routes.Distance["Remote"]}, route={string.Join("->",routes.PathTo("Remote"))}; cable components={forest.Components}");
if(routes.Distance["Remote"]!=0||forest.Components!=2)throw new Exception("Model distinction failed.");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


Remote delivery=0, route=HQ->Remote; cable components=2


## Project Connection

```{index} Project Connection
```

Specify one possible optimization model for your project. State objective, constraints, weight units, direction, boundary policy, and a proof sketch or a counterexample that limits the greedy rule.

## Submit

```{index} Submit
```

Submit code, verified output, and reasoning through the assignment panel when available.